In [1]:
import os
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import StandardScaler

In [5]:


DATASET_PATH = r"../DATASET"

csv_files = [
    os.path.join(DATASET_PATH, f)
    for f in os.listdir(DATASET_PATH)
    if f.endswith(".csv")
]

df = pd.DataFrame()

for file in csv_files:
    print("Loading:", file)

    for chunk in pd.read_csv(file, chunksize=200000):
        df = pd.concat([df, chunk], ignore_index=True)

print("Final shape:", df.shape)

Loading: ../DATASET\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: ../DATASET\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: ../DATASET\Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: ../DATASET\Monday-WorkingHours.pcap_ISCX.csv
Loading: ../DATASET\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: ../DATASET\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: ../DATASET\Tuesday-WorkingHours.pcap_ISCX.csv
Loading: ../DATASET\Wednesday-workingHours.pcap_ISCX.csv
Final shape: (2830743, 79)


In [6]:
columns_to_remove = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Source Port",
    "Destination Port",
    "Protocol"
]

df = df.drop(columns=columns_to_remove, errors='ignore')

print("Remaining columns:", df.shape)

Remaining columns: (2830743, 79)


In [8]:
df.columns = df.columns.str.strip()

In [9]:
df['Label'] = df['Label'].apply(lambda x: 0 if x == "BENIGN" else 1)

print(df['Label'].value_counts())

Label
0    2273097
1     557646
Name: count, dtype: int64


In [10]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [11]:
df = df.dropna()

print("Dataset shape after removing NaNs:", df.shape)

Dataset shape after removing NaNs: (2827876, 79)


In [12]:
X = df.drop("Label", axis=1)
y = df["Label"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

Feature shape: (2827876, 78)
Label shape: (2827876,)


In [13]:
corr_matrix = X.corr().abs()

In [14]:
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

In [15]:
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]

print("Highly correlated features:", len(to_drop))

Highly correlated features: 31


In [16]:
X = X.drop(columns=to_drop)

print("Features after correlation filtering:", X.shape)

Features after correlation filtering: (2827876, 47)


In [17]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(2827876, 47)


In [18]:
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [19]:
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [20]:
processed_df = X_scaled.copy()
processed_df["Label"] = y.values

processed_df.to_csv("processed_cicids2017.csv", index=False)

print("Preprocessed dataset saved.")

Preprocessed dataset saved.
